In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np
import json
import os

drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/medical_protocols/'
TRANSLATED_PATH = os.path.join(BASE_PATH, 'translated_english_llama/')
TOP2VEC_PATH = os.path.join(BASE_PATH, 'top2vec/')

print("✅ Drive connected. Paths initialized.")

Mounted at /content/drive
✅ Drive connected. Paths initialized.


In [ ]:
json_file_path = os.path.join(TRANSLATED_PATH, 'drugs_Акушерский_сепсис.json')

manual_truth = [
    "ceftriaxone", "metronidazole", "gentamicin", "piperacillin/tazobactam",
    "amikacin", "vancomycin", "linezolid", "imipenem", "meropenem",
    "norepinephrine", "dopamine", "magnesium sulfate", "heparin", "insulin"
]

try:
    with open(json_file_path, 'r', encoding='utf-8') as f:
        auto_extracted = [d.lower() for d in json.load(f) if isinstance(d, str)]

    tp = set(manual_truth).intersection(set(auto_extracted))
    fp = set(auto_extracted) - set(manual_truth)
    fn = set(manual_truth) - set(auto_extracted)

    recall = len(tp) / len(manual_truth)
    precision = len(tp) / len(auto_extracted)

    print(f"--- TABLE 3.1: NER VALIDATION (Protocol: Obstetric Sepsis) ---")
    print(f"Recall: {recall:.2%} (Полнота извлечения важных лекарств)")
    print(f"Precision: {precision:.2%} (Точность - отсутствие шума)")
    print(f"\nНайдено верно: {list(tp)[:5]}...")
    print(f"Пропущено (False Negatives): {list(fn)}")
    print(f"Шум (False Positives): {list(fp)[:5]}...")

except FileNotFoundError:
    print(f"❌ Файл не найден по пути: {json_file_path}. Проверьте папку на Google Drive.")

--- TABLE 3.1: NER VALIDATION (Protocol: Obstetric Sepsis) ---
Recall: 64.29% (Полнота извлечения важных лекарств)
Precision: 20.45% (Точность - отсутствие шума)

Найдено верно: ['dopamine', 'metronidazole', 'vancomycin', 'amikacin', 'gentamicin']...
Пропущено (False Negatives): ['meropenem', 'linezolid', 'magnesium sulfate', 'piperacillin/tazobactam', 'imipenem']
Шум (False Positives): ['rifampicin', 'albuterol', 'nafcillin', 'none mentioned in this section.', 'doxorubicin']...


In [ ]:
fidelity_data = [
    {"RU": "АД сист ≤ 100 мм рт.ст.", "EN": "Systolic BP ≤ 100 mmHg", "Score": 3},
    {"RU": "ЧД ≥ 22 в мин", "EN": "Respiratory Rate ≥ 22 per min", "Score": 3},
    {"RU": "нарушение сознания", "EN": "altered consciousness", "Score": 3},
    {"RU": "лактат плазмы > 2 ммоль/л", "EN": "plasma lactate > 2 mmol/l", "Score": 3},
    {"RU": "гистерэктомия без придатков", "EN": "hysterectomy without appendages", "Score": 3},
    {"RU": "сепсис-индуцированная гипотензия", "EN": "sepsis-induced hypotension", "Score": 3},
    {"RU": "полиорганная недостаточность", "EN": "multi-organ failure", "Score": 3},
    {"RU": "шкала qSOFA", "EN": "qSOFA score", "Score": 3},
    {"RU": "внутривенное введение", "EN": "intravenous administration", "Score": 3},
    {"RU": "мониторинг диуреза", "EN": "urine output monitoring", "Score": 3}
]

fidelity_df = pd.DataFrame(fidelity_data)
print("--- TABLE 3.2: TRANSLATION FIDELITY AUDIT ---")
print(fidelity_df.to_string(index=False))
print(f"\nFinal Expert Fidelity Score: {fidelity_df['Score'].mean()}/3.0")

--- TABLE 3.2: TRANSLATION FIDELITY AUDIT ---
                              RU                              EN  Score
         АД сист ≤ 100 мм рт.ст.          Systolic BP ≤ 100 mmHg      3
                   ЧД ≥ 22 в мин   Respiratory Rate ≥ 22 per min      3
              нарушение сознания           altered consciousness      3
       лактат плазмы > 2 ммоль/л       plasma lactate > 2 mmol/l      3
     гистерэктомия без придатков hysterectomy without appendages      3
сепсис-индуцированная гипотензия      sepsis-induced hypotension      3
    полиорганная недостаточность             multi-organ failure      3
                     шкала qSOFA                     qSOFA score      3
           внутривенное введение      intravenous administration      3
              мониторинг диуреза         urine output monitoring      3

Final Expert Fidelity Score: 3.0/3.0


In [ ]:
csv_topic_path = os.path.join(TOP2VEC_PATH, 'top2vec_topics_EN.csv')

try:
    topics_df = pd.read_csv(csv_topic_path)

    human_labels = {
        0: "General Obstetric Care & Management",
        1: "Prenatal Diagnostics & Screening",
        2: "Reproductive Health & Contraception",
        3: "Surgical & Gynecological Procedures",
        4: "Antenatal Monitoring Protocols",
        5: "Clinical Treatment Guidelines",
        6: "Patient Care Standardization",
        7: "Laboratory & Diagnostic Testing",
        8: "Clinical Assessment Frameworks",
        9: "Invasive Interventions & Risks"
    }

    topics_df['Clinical_Domain'] = topics_df['topic_id'].map(human_labels)

    print("--- TABLE 3.3: SEMANTIC TOPIC IDENTIFICATION ---")
    print(topics_df[['topic_id', 'Clinical_Domain', 'size']].head(10).to_string(index=False))

except FileNotFoundError:
    print(f"❌ Файл CSV не найден по пути: {csv_topic_path}")

--- TABLE 3.3: SEMANTIC TOPIC IDENTIFICATION ---
 topic_id                     Clinical_Domain  size
        0 General Obstetric Care & Management  1214
        1    Prenatal Diagnostics & Screening   164
        2 Reproductive Health & Contraception   156
        3 Surgical & Gynecological Procedures   134
        4      Antenatal Monitoring Protocols   114
        5       Clinical Treatment Guidelines    87
        6        Patient Care Standardization    53
        7     Laboratory & Diagnostic Testing    44
        8      Clinical Assessment Frameworks    38
        9      Invasive Interventions & Risks    25
